# 006 · Unpacking

> **Syntax › Collections** · Python 3.14

*Unpacking* assigns the items of any iterable to several names in one statement: `a, b = pair`. It is the Python counterpart of C# deconstruction (`var (a, b) = pair;`), but it works on **any** iterable and has extra tricks like starred targets and `*`/`**` spreading.

**In this notebook you will learn:**

- basic unpacking and the one-line swap
- starred targets: `first, *rest = items`
- nested unpacking and unpacking in `for` loops
- ignoring values with `_`
- `*` and `**` to spread iterables and dicts into calls and literals
- what happens when the counts don't match

## C# mental model

| C# | Python |
|---|---|
| `var (a, b) = (1, 2);` | `a, b = 1, 2` |
| `(a, b) = (b, a);` | `a, b = b, a` |
| `var (_, y) = point;` | `_, y = point` |
| `Deconstruct(out ...)` method | any iterable unpacks (lists, tuples, strings, generators…) |
| list pattern `[var first, .. var rest]` | `first, *rest = items` |
| `[.. a, .. b]` (C# 12 spread) | `[*a, *b]` |

In C#, deconstruction requires a tuple or a type with a `Deconstruct` method. In Python, **anything iterable** can be unpacked — the right-hand side is simply iterated.

## Basic unpacking

```csharp
// C#
var point = (3, 4);
var (x, y) = point;
```

In [ ]:
point = (3, 4)
x, y = point
print(x, y)

r, g, b = [255, 128, 0]          # from a list
first, second, third = "abc"     # from a string — any iterable works
print(r, g, b, first, second, third)

Unpacking a dict iterates its **keys**. Use `.items()` or `.values()` if you want something else.

In [ ]:
config = {"host": "localhost", "port": 8080}
k1, k2 = config
print(k1, k2)

(name1, val1), (name2, val2) = config.items()
print(name1, val1, name2, val2)

## Swapping without a temp variable

The right-hand side is evaluated into a tuple first, then unpacked — so swapping is a one-liner, identical in spirit to C#'s tuple swap.

```csharp
// C#
(a, b) = (b, a);
```

In [ ]:
a, b = 1, 2
a, b = b, a
print(a, b)

## Count mismatches raise `ValueError`

Without a starred target, the number of names must match the number of items exactly. C# catches this at compile time for tuples; Python finds out at runtime.

In [ ]:
try:
    x, y = [1, 2, 3]
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

try:
    x, y, z = [1, 2]
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

⚠️ **Gotcha:** unpacking a string you expected to be a pair. `a, b = "hi"` works (two characters!), but `a, b = "hello"` fails — the bug only surfaces when the data changes.

In [ ]:
a, b = "hi"
print(a, b)     # 'h' 'i' — probably not what was intended

## Starred targets: `*rest`

One target may be prefixed with `*`. It collects all "leftover" items into a **list** (even if the source was a tuple or string). This is similar to the slice pattern `..` in C# list patterns.

```csharp
// C# 11 list patterns
if (items is [var first, .. var middle, var last]) { ... }
```

In [ ]:
numbers = [1, 2, 3, 4, 5]

first, *rest = numbers
print(first, rest)

*init, last = numbers
print(init, last)

head, *middle, tail = numbers
print(head, middle, tail)

The starred name may receive an empty list, but the non-starred targets must still be filled:

In [ ]:
a, *b = [1]
print(a, b)           # 1 []

try:
    a, *b, c = [1]
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

Only one starred target is allowed per level — otherwise the split would be ambiguous. This is a `SyntaxError` at compile time, so it's shown here rather than executed:

```python
*a, *b = [1, 2, 3]   # SyntaxError: multiple starred expressions in assignment
```

## Ignoring values with `_`

By convention `_` names a value you don't care about — like C#'s discard `_`. Unlike in C#, Python's `_` is a **real variable** that gets assigned (and overwritten each time). Combine it with `*` to ignore many values.

```csharp
// C#
var (_, _, z) = (1, 2, 3);
```

In [ ]:
_, _, z = (1, 2, 3)
print(z)

record = ("alice", "admin", "alice@example.com", "2026-01-01", "active")
username, *_, status = record
print(username, status)
print(_)      # _ really is assigned — it holds the ignored middle values

## Nested unpacking

Targets can mirror the structure of the data. C# supports this too with nested deconstruction.

```csharp
// C#
var (name, (lat, lon)) = ("home", (51.5, -0.1));
```

In [ ]:
location = ("home", (51.5, -0.1))
name, (lat, lon) = location
print(name, lat, lon)

(a, b), [c, *d] = [(1, 2), [3, 4, 5]]   # () and [] are interchangeable on the left
print(a, b, c, d)

## Unpacking in `for` loops

Unpacking happens automatically in `for` targets — this is why `for k, v in d.items()` and `for i, x in enumerate(xs)` work.

```csharp
// C#
foreach (var (name, score) in results) { ... }
```

In [ ]:
results = [("alice", 92), ("bob", 78)]
for name, score in results:
    print(f"{name}: {score}")

for i, (name, score) in enumerate(results, start=1):   # nested in a loop
    print(i, name, score)

for first, *others in [[1, 2, 3], [4], [5, 6]]:
    print(first, others)

## `*` in calls: spreading an iterable into positional arguments

Prefixing an argument with `*` spreads an iterable into **positional** arguments. C# has no direct equivalent (the closest is passing an array to a `params` parameter).

In [ ]:
def describe(name, age, city):
    return f"{name}, {age}, from {city}"

row = ["alice", 30, "Springfield"]
print(describe(*row))            # same as describe("alice", 30, "Springfield")

print(*[1, 2, 3], sep=" | ")     # print takes any number of positional args

## `**` in calls: spreading a dict into keyword arguments

`**mapping` spreads a dict into **keyword** arguments — keys become parameter names. This is common for passing configuration through. (Collecting arguments with `*args` and `**kwargs` on the receiving side is covered in *Functions › 003 · Args and Kwargs*.)

In [ ]:
options = {"name": "bob", "age": 25, "city": "Shelbyville"}
print(describe(**options))

partial = {"age": 40, "city": "Ogdenville"}
print(describe("carol", **partial))   # mix positional and spread

try:
    describe(**{"name": "x", "age": 1, "colour": "red"})
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

## `*` and `**` in literals

The same operators spread items into new list, tuple, set and dict literals. C# 12 collection expressions offer a spread `..` for collections, but not for dictionaries.

```csharp
// C# 12
int[] a = [1, 2];
int[] b = [3, 4];
int[] all = [.. a, .. b, 5];
```

In [ ]:
a = [1, 2]
b = (3, 4)

print([*a, *b, 5])              # list
print((*a, *b))                 # tuple
print({*a, *b, 1})              # set (duplicates removed)
print([*"hi", *range(3)])       # any iterable

`{**d1, **d2}` merges dicts — later keys win, exactly like `d1 | d2` *(see Collections › 003 · Dictionaries)*. It's handy when you also want to add literal keys.

In [ ]:
defaults = {"theme": "light", "lang": "en"}
user = {"theme": "dark"}

settings = {**defaults, **user, "beta": True}
print(settings)

## Starred assignment vs function `*args`

The same `*` symbol means "collect" on the **left** side of `=` (or in a parameter list) and "spread" on the **right** side (in a call or literal):

In [ ]:
values = [10, 20, 30, 40]

first, *rest = values          # collect: rest is a list
again = [first, *rest]         # spread: rebuild the list
print(rest, again == values)

## Summary

| Task | C# | Python |
|---|---|---|
| Deconstruct | `var (a, b) = t;` | `a, b = t` |
| Swap | `(a, b) = (b, a);` | `a, b = b, a` |
| Discard | `var (_, y) = t;` | `_, y = t` |
| Head / tail | `[var h, .. var t]` (pattern) | `h, *t = items` |
| Nested | `var (n, (x, y)) = v;` | `n, (x, y) = v` |
| In foreach | `foreach (var (k, v) in d)` | `for k, v in d.items():` |
| Spread into call | `params` array | `f(*items)`, `f(**options)` |
| Spread into collection | `[.. a, .. b]` | `[*a, *b]` |
| Merge dicts | manual loop | `{**d1, **d2}` |
| Wrong count | compile error | `ValueError` at runtime |

## Exercises

1. Given `scores = [88, 92, 79, 95, 60]`, sort it and unpack the lowest score, the highest score, and everything in between.
2. Given `rows = [("alice", ("London", "UK")), ("bob", ("Paris", "FR"))]`, loop over the rows and print `"<name> lives in <city>"` using nested unpacking.
3. Call `pow(base, exp)` using a list `[2, 10]` spread into the call, then using a dict spread.
4. Build one list containing the items of `a = [1, 2]`, then `0`, then the items of `b = (3, 4)` — in one expression.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1
scores = [88, 92, 79, 95, 60]
lowest, *middle, highest = sorted(scores)   # 60, [79, 88, 92], 95

# 2
rows = [("alice", ("London", "UK")), ("bob", ("Paris", "FR"))]
for name, (city, _country) in rows:
    print(f"{name} lives in {city}")

# 3
print(pow(*[2, 10]))                  # 1024
print(pow(**{"base": 2, "exp": 10}))  # 1024 — pow's parameters are named base and exp

# 4
a, b = [1, 2], (3, 4)
combined = [*a, 0, *b]                # [1, 2, 0, 3, 4]
```

</details>